In [ ]:
# Generate comprehensive report
report = {
    'summary': {
        'total_pioneers': len(df),
        'exact_duplicate_names': len(exact_duplicates),
        'similar_names_found': len(similar_pairs),
        'word_order_variants': len(word_order_variants),
        'duplicate_years': len(duplicate_years),
        'malformed_names': len(malformed_names)
    },
    'details': {
        'exact_duplicates': [],
        'similar_names': similar_pairs,
        'word_variants': word_order_variants,
        'year_duplicates': [],
        'malformed': malformed_names
    }
}

# Build details for exact duplicates
for name, count in exact_duplicates.items():
    matching_rows = df[df['name'] == name]
    entries = []
    for idx, row in matching_rows.iterrows():
        entries.append({
            'filename': row['filename'],
            'birth': row['birth_year'],
            'death': row['death_year']
        })
    report['details']['exact_duplicates'].append({
        'name': name,
        'count': count,
        'entries': entries
    })

# Build details for year duplicates
for year_key, count in duplicate_years.items():
    matching_rows = df_with_years[df_with_years['year_key'] == year_key]
    entries = []
    for idx, row in matching_rows.iterrows():
        entries.append({
            'name': row['name'],
            'filename': row['filename']
        })
    report['details']['year_duplicates'].append({
        'years': year_key,
        'count': count,
        'entries': entries
    })

# Print formatted report
print("=" * 80)
print("COMPREHENSIVE PIONEER DUPLICATE ANALYSIS REPORT")
print("=" * 80)
print(f"\n📊 SUMMARY STATISTICS:")
print(f"  Total pioneer records analyzed: {report['summary']['total_pioneers']}")
print(f"  Exact duplicate names: {report['summary']['exact_duplicate_names']}")
print(f"  Similar names detected: {report['summary']['similar_names_found']}")
print(f"  Word order variants: {report['summary']['word_order_variants']}")
print(f"  Duplicate year combinations: {report['summary']['duplicate_years']}")
print(f"  Potentially malformed names: {report['summary']['malformed_names']}")

print(f"\n" + "=" * 80)
print("DETAILED FINDINGS")
print("=" * 80)

if len(report['details']['exact_duplicates']) > 0:
    print(f"\n❌ EXACT DUPLICATE NAMES ({len(report['details']['exact_duplicates'])}):")
    for item in report['details']['exact_duplicates']:
        print(f"\n  Name: '{item['name']}' (appears {item['count']} times)")
        for entry in item['entries']:
            print(f"    • {entry['filename']}: ({entry['birth']}-{entry['death']})")
else:
    print(f"\n✅ EXACT DUPLICATE NAMES: None found")

if len(report['details']['similar_names']) > 0:
    print(f"\n⚠️  SIMILAR NAMES ({len(report['details']['similar_names'])}):")
    for item in sorted(report['details']['similar_names'], key=lambda x: x['similarity'], reverse=True):
        print(f"\n  \"{item['name1']}\" vs \"{item['name2']}\"")
        print(f"    Similarity: {item['similarity']:.1%}")
        print(f"    • {item['file1']}: ({item['birth1']}-{item['death1']})")
        print(f"    • {item['file2']}: ({item['birth2']}-{item['death2']})")
else:
    print(f"\n✅ SIMILAR NAMES: None found above threshold")

if len(report['details']['word_variants']) > 0:
    print(f"\n🔄 WORD ORDER VARIANTS ({len(report['details']['word_variants'])}):")
    for item in report['details']['word_variants']:
        print(f"\n  \"{item['name1']}\" {item['years1']} vs \"{item['name2']}\" {item['years2']}")
        print(f"    • {item['file1']} vs {item['file2']}")
else:
    print(f"\n✅ WORD ORDER VARIANTS: None found")

if len(report['details']['year_duplicates']) > 0:
    print(f"\n📅 DUPLICATE YEAR COMBINATIONS ({len(report['details']['year_duplicates'])}):")
    for item in report['details']['year_duplicates']:
        print(f"\n  Years {item['years']} (appears {item['count']} times):")
        for entry in item['entries']:
            print(f"    • {entry['name']} ({entry['filename']})")
else:
    print(f"\n✅ DUPLICATE YEAR COMBINATIONS: None found")

print(f"\n" + "=" * 80)
print("END OF REPORT")
print("=" * 80)

## 9. Generate Comprehensive Duplicate Report

In [ ]:
def check_name_validity(name):
    """Check if a name is properly formatted."""
    issues = []
    
    if not name or name == 'N/A':
        issues.append('Missing name')
    elif len(name) < 2:
        issues.append('Very short name (< 2 chars)')
    elif len(name.split()) < 1:
        issues.append('No word separators')
    elif name.count('.') > 5:
        issues.append('Too many periods (likely malformed)')
    elif bool(re.search(r'[<>{}[\]\\]', name)):
        issues.append('Contains suspicious characters')
    elif name.isupper() and len(name) > 5:
        issues.append('All uppercase (possible formatting issue)')
    
    # Check for common issues
    if name.startswith('Brother') and len(name.split()) < 2:
        issues.append('Incomplete "Brother" name')
    if name.startswith('Dr ') or name.startswith('Dr. '):
        if len(name.split()) < 3:
            issues.append('Incomplete doctor name')
    
    return issues

# Validate all names
malformed_names = []
for idx, row in df.iterrows():
    issues = check_name_validity(row['name'])
    if issues:
        malformed_names.append({
            'name': row['name'],
            'filename': row['filename'],
            'issues': issues
        })

print(f"⚠️  POTENTIALLY MALFORMED NAMES: {len(malformed_names)}")
if len(malformed_names) > 0:
    for entry in malformed_names:
        print(f"\n  '{entry['name']}' ({entry['filename']})")
        for issue in entry['issues']:
            print(f"    - {issue}")
else:
    print("  ✓ All names appear well-formatted!")

## 8. Validate Name Format

In [ ]:
# Filter out entries with missing years
df_with_years = df[(df['birth_year'].notna()) & (df['death_year'].notna())].copy()

# Create year combination key
df_with_years['year_key'] = df_with_years['birth_year'] + '-' + df_with_years['death_year']

# Find duplicate year combinations
year_counts = df_with_years['year_key'].value_counts()
duplicate_years = year_counts[year_counts > 1]

print(f"🔍 DUPLICATE BIRTH/DEATH YEAR COMBINATIONS: {len(duplicate_years)}")
if len(duplicate_years) > 0:
    for year_key, count in duplicate_years.items():
        matching_rows = df_with_years[df_with_years['year_key'] == year_key]
        print(f"\n  Years {year_key} appears {count} times:")
        for idx, row in matching_rows.iterrows():
            print(f"    - {row['name']:<35} ({row['filename']})")
else:
    print("  ✓ No duplicate year combinations found!")

## 7. Detect Duplicate Birth/Death Year Combinations

In [ ]:
def calculate_similarity(name1, name2):
    """Calculate similarity between two names using SequenceMatcher."""
    return SequenceMatcher(None, name1.lower(), name2.lower()).ratio()

# Find similar names with high similarity score
SIMILARITY_THRESHOLD = 0.75
similar_pairs = []
names_list = df['name'].unique().tolist()

for i, name1 in enumerate(names_list):
    for name2 in names_list[i+1:]:
        if name1 == name2:
            continue
        
        similarity = calculate_similarity(name1, name2)
        if similarity >= SIMILARITY_THRESHOLD:
            # Get the rows for both names
            row1 = df[df['name'] == name1].iloc[0]
            row2 = df[df['name'] == name2].iloc[0]
            
            similar_pairs.append({
                'name1': name1,
                'name2': name2,
                'similarity': similarity,
                'file1': row1['filename'],
                'file2': row2['filename'],
                'birth1': row1['birth_year'],
                'death1': row1['death_year'],
                'birth2': row2['birth_year'],
                'death2': row2['death_year']
            })

# Also check for word ordering variations (e.g., "Joseph Ellet" vs "Ellet Joseph")
word_order_variants = []
for i, name1 in enumerate(names_list):
    for name2 in names_list[i+1:]:
        words1 = set(name1.lower().split())
        words2 = set(name2.lower().split())
        
        # If they have the same words but in different order
        if len(words1) >= 2 and words1 == words2:
            row1 = df[df['name'] == name1].iloc[0]
            row2 = df[df['name'] == name2].iloc[0]
            
            word_order_variants.append({
                'name1': name1,
                'name2': name2,
                'file1': row1['filename'],
                'file2': row2['filename'],
                'years1': f"({row1['birth_year']}-{row1['death_year']})",
                'years2': f"({row2['birth_year']}-{row2['death_year']})"
            })

print(f"🔍 SIMILAR NAMES (similarity ≥ {SIMILARITY_THRESHOLD}): {len(similar_pairs)}")
if len(similar_pairs) > 0:
    for i, pair in enumerate(sorted(similar_pairs, key=lambda x: x['similarity'], reverse=True), 1):
        print(f"\n  {i}. \"{pair['name1']}\" vs \"{pair['name2']}\" (similarity: {pair['similarity']:.2%})")
        print(f"     File 1: {pair['file1']} ({pair['birth1']}-{pair['death1']})")
        print(f"     File 2: {pair['file2']} ({pair['birth2']}-{pair['death2']})")
else:
    print("  ✓ No similar names found above threshold!")

print(f"\n📄 WORD ORDER VARIANTS: {len(word_order_variants)}")
if len(word_order_variants) > 0:
    for i, pair in enumerate(word_order_variants, 1):
        print(f"\n  {i}. \"{pair['name1']}\" {pair['years1']} vs \"{pair['name2']}\" {pair['years2']}")
        print(f"     Files: {pair['file1']} vs {pair['file2']}")
else:
    print("  ✓ No word order variants found!")

## 6. Find Similar Names Using String Matching

In [ ]:
# Find exact duplicate names
name_counts = df['name'].value_counts()
exact_duplicates = name_counts[name_counts > 1]

print(f"🔍 EXACT DUPLICATE NAMES: {len(exact_duplicates)}")
if len(exact_duplicates) > 0:
    for name, count in exact_duplicates.items():
        matching_rows = df[df['name'] == name]
        print(f"\n  '{name}' appears {count} times:")
        for idx, row in matching_rows.iterrows():
            print(f"    - {row['filename']}: ({row['birth_year']}-{row['death_year']})")
else:
    print("  ✓ No exact duplicate names found!")

## 5. Identify Exact Duplicate Names

In [ ]:
# Create DataFrame
df = pd.DataFrame(pioneers_data)
df = df.sort_values('name').reset_index(drop=True)

print(f"Pioneer Database:")
print(f"  Total records: {len(df)}")
print(f"\nDataFrame shape: {df.shape}")
print(f"\nFirst 10 pioneers:")
display(df.head(10))

print(f"\nData quality check:")
print(f"  Names with 'N/A': {(df['name'] == 'N/A').sum()}")
print(f"  Missing birth years: {df['birth_year'].isna().sum()}")
print(f"  Missing death years: {df['death_year'].isna().sum()}")

## 4. Build Pioneer Database

In [ ]:
def extract_frontmatter(file_path):
    """Extract frontmatter fields from a markdown file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Extract frontmatter (between --- markers)
        fm_match = re.search(r'^---\n([\s\S]*?)\n---', content, re.MULTILINE)
        if not fm_match:
            return None
        
        frontmatter = fm_match.group(1)
        
        # Extract name
        name_match = re.search(r'name:\s*["\']?([^"\'\n]+)["\']?', frontmatter)
        name = name_match.group(1).strip() if name_match else 'N/A'
        
        # Extract birth year
        birth_match = re.search(r'birth:\s*(\d+)', frontmatter)
        birth = birth_match.group(1) if birth_match else None
        
        # Extract death year
        death_match = re.search(r'death:\s*(\d+)', frontmatter)
        death = death_match.group(1) if death_match else None
        
        return {
            'name': name,
            'birth': birth,
            'death': death
        }
    except Exception as e:
        print(f"Error reading {file_path.name}: {e}")
        return None

# Extract data from all files
pioneers_data = []

for md_file in md_files:
    fm_data = extract_frontmatter(md_file)
    if fm_data:
        pioneers_data.append({
            'filename': md_file.name,
            'name': fm_data['name'],
            'birth_year': fm_data['birth'],
            'death_year': fm_data['death']
        })

print(f"✓ Extracted data from {len(pioneers_data)} pioneer files")
print(f"\nSample data:")
for item in pioneers_data[:5]:
    print(f"  {item['name']:<35} ({item['birth_year']}-{item['death_year']}) - {item['filename']}")

## 3. Extract Frontmatter Data

In [ ]:
# Define the directory containing pioneer markdown files
pioneers_dir = Path(r'c:\Users\weddi\OneDrive\Desktop\pioneers\src\content\pioneers')

# Search for all .md files, excluding prophecy charts and non-pioneer files
md_files = []
excluded_patterns = ['prophecy', 'henry-mead.jpg', 'early-tract']

for md_file in sorted(pioneers_dir.glob('*.md')):
    filename = md_file.name
    # Skip excluded files
    if any(pattern in filename.lower() for pattern in excluded_patterns):
        continue
    md_files.append(md_file)

print(f"✓ Found {len(md_files)} pioneer markdown files")
print(f"Sample files: {[f.name for f in md_files[:5]]}")

## 2. Search for Markdown Files in Directory

In [ ]:
import os
import re
import json
from pathlib import Path
from collections import defaultdict
import pandas as pd
from difflib import SequenceMatcher
import warnings
warnings.filterwarnings('ignore')

## 1. Import Required Libraries

# Pioneer Duplicate Analysis
Comprehensive analysis of pioneer markdown files to identify duplicate entries, similar names, and data inconsistencies.